**Benchmarking Script**

Warmup steps: 5 | Measurement steps: 10 | Device: MPS

| Model | Forward (ms) | Forward+Backward (ms) | Full (ms) |
|---|---|---|---|
| small | 99.9 ± 0.2 | 295.4 ± 0.1 | 313.4 ± 0.2 |
| medium | 291.8 ± 0.2 | 870.4 ± 0.4 | 935.5 ± 1.6 |
| large | 617.1 ± 3.3 | 1893.9 ± 29.9 | OOM/ERR |

The standard deviations (see above) are small. Decided not to run the XL and 10B models since they likely will not run on laptop.

Now I ran with 0 warm up steps (my laptop crashed with large model, so I omitted this):

Warmup steps: 0 | Measurement steps: 10 | Device: MPS

| Model | Forward (ms) | Forward+Backward (ms) | Full (ms) |
|---|---|---|---|
| small | 108.6 ± 26.2 | 300.6 ± 12.1 | 319.1 ± 11.2 |
| medium | 296.3 ± 10.0 | 877.0 ± 12.6 | 943.6 ± 15.1 |

As can be seen the times are slightly longer, likely due to cold cache.

And finally running with one warmup step:

Warmup steps: 1 | Measurement steps: 10 | Device: MPS

| Model | Forward (ms) | Forward+Backward (ms) | Full (ms) |
|---|---|---|---|
| small | 99.7 ± 0.1 | 294.8 ± 0.1 | 313.1 ± 0.5 |
| medium | 291.5 ± 0.4 | 867.5 ± 0.1 | 933.7 ± 1.0 |

Even just one warmup step seems to make a difference.

**PyTorch Profiling**

Note: I am not using Nsys but rather torch.profile for compatibility with my macbook.

(a) Total time spent for forward is:

| Model | Batch Size | Time (ms) |
|---|---|---|
| small | 256 | 53.80 |
| small | 512 | 115.05 |
| medium | 256 | 145.75 |
| medium | 512 | 319.51 |

For batch size 512, the times without profiling for small (99.9ms) and medium (291.8ms) are slightly smaller as expected.

(b) The `aten::bmm` (batched matrix multiply) kernel takes most time on the forwards pass, consuming ~30% of runtime regardless of model (small/medium). This kernel is called 217 times (medium model) and 109 times (small model). While the forwards pass is dominated by `bmm`, the backwards pass also has `aten::sum` as a significant component, which makes sense since the autograd engine performs lots of summations. (*weird behaviour seen with medium model and 1024 context length: `sum` becomes negligible and `bmm` jumps to ~80%*).

(c) `einsum`, `mean`, `permute`, `max`, `mul`

(d) Compared to inference, when performing a full (forwards/backwards/optimizer) pass, the fraction of time spent on `bmm` falls slightly and time spent on `addcdiv` and `addcmul` rises slightly (which are elementwise operations directly related to steps within the AdamW algorithm).

(e) To perform this comparison, markers were added within the `scaled_dot_product_attention` function. The FLOPs was estimated using analytical formulas. Although the FLOPs ratio (mul FLOPs / softmax FLOPs) is ~85, the runtime ratio is just ~1.3x suggesting softmax is memory bound and that it takes a long time despite relatively little computation.

**NSys Profiling on AWS Cluster**

(a) (skipped)

(b) The `xmma_gemm` kernels take most time on forwards pass (matrix multiply kernels). Using the medium model for reference:

| Context length | Kernel | Runtime % | No. calls |
| -- | -- | -- | -- |
| 256 | sm90_xmma_gemm...tn...64x128x32 | 23.2 | 120 |
| 512 | sm90_xmma_gemm...tn...128x128x32 | 25.2 | 169 |
| 1024 | sm90_xmma_gemm...tn...128x128x32 | 16.2 | 168 |

Note: for 256, there are also 48 calls for `sm90_xmma_gemm...tn...128x256x32` (which added to the 120 calls shown above, gives 168 calls). I therefore think the same number of kernel calls are happening, but depending upon context length the exact kernels used differ.

Also note: there are different kernel transpose variants (`nn`, `tn`, `nt`) which complicate matters. In forwards pass the `tn` dominates.

For 512-context medium, in backwards pass, no single `xma_gemm` kernel takes a large (>10%) runtime, although summing the transpose variants shows matrix multiplication consumes ~33% of runtime in both forwards and backwards passes. Therefore the matmul kernels are more varied in backwards pass, likely a direct consequence of needing gradients with respect to both inputs and weights. The `MulFunctor` kernel jumps from 1.620ms (forward) to 7.387ms (backward), a 4.6× increase, because the backward pass adds gradient computations for SwiGLU, RoPE, and residual connections, all of which involve elementwise multiplications.

(c) Numbers for medium model: `vectorized_elementwise_kernel BinaryFunctor` (elementwise multiply), 6.1% at `ctx=256`, 7.1% at `ctx=512`, 5.5% at `ctx=1024`; `reduce_kernel MaxOps`, finds max for numerical stability (3.3% to 4.8%); `exp_kernel`, computes exponentials (2.2% to 7.4%)

(d) Using 512-context medium, the fraction of time spent on matrix multiplication decreases slightly from ~33% to ~29%. In addition, new kernels such as `addcdiv_cuda_kernel` and `addcmul_cuda_kernel` appear.

(e) (skipped)

**Mixed Precision Accumulation**

FP32 gives the most accurate answer. FP16 gives ~5% error. The third and fourth runs give same answer revealing FP16 promotes to FP32. This is slightly worse than pure FP32 because the FP16 error manifests before the promotion occurs.

**Benchmarking Mixed Precision**

| Component | dtype | Reason |
|---|---|---|
| Model parameters | float32 | autocast never changes stored parameter dtypes |
| fc1 output | float16 | matmul is on autocast whitelist |
| LayerNorm output | float32 | reduction op, excluded from whitelist for numerical stability |
| logits (fc2 output) | float16 | matmul is on autocast whitelist |
| loss | float32 | reduction op, excluded from whitelist for numerical stability |
| gradients | float32 | always float32 regardless of autocast |
 
---
 
**EXPLANATION**
 
**Model parameters** remain float32 throughout — autocast only affects computation dtype, not storage dtype. The float32 weights are temporarily cast to float16 for eligible operations but the stored values are unchanged.
 
**fc1 output is float16** because `nn.Linear` (matmul) is on PyTorch's autocast whitelist of eligible operations. The float32 weights are cast to float16 for the computation and the output is float16.
 
**LayerNorm output is float32** because LayerNorm involves reductions (mean, variance) that are numerically sensitive and prone to overflow/underflow in float16. PyTorch's autocast whitelist explicitly excludes LayerNorm for this reason. This requires a dtype cast from float16 → float32, which has a small but measurable cost (visible as `direct_copy_kernel` in nsys profiles).
 
**Logits (fc2 output) are float16** — fc2 is another matmul on the whitelist. Even though its input from LayerNorm is float32, autocast casts it back to float16 for the matmul, giving float16 output.
 
**Loss is float32** — CrossEntropyLoss involves reductions (sum, log, exp) kept in float32 for numerical stability, even though the input logits are float16.
 
**Gradients are float32** — gradients are always computed and stored in float32 regardless of autocast. This is critical for training stability — float16 gradients could underflow to zero for small gradient values, causing parameters to stop updating.
 
---
 
**FP16 VS BF16**
 
If using `dtype=torch.bfloat16` instead of `torch.float16`, LayerNorm and other reductions can safely run in bfloat16 because bfloat16 has the same dynamic range as float32 (8 exponent bits vs 5 for float16). This reduces the number of dtype conversions needed, improving performance at the cost of slightly lower mantissa precision (7 bits vs 10 for float16).

**Benchmarking Mixed Precision**

Running BF16 mixed precision against FP32 across small, medium, and large model sizes, we observed modest speedups of 10-20% rather than the theoretical ~2× improvement. 

This is explained by three factors: 

- the H100 already uses TF32 tensor cores for FP32 matmuls by default (visible in nsys kernel names as `f32f32_tf32f32_f32`), meaning the FP32 baseline is already faster than true scalar FP32
- only ~33% of forward pass kernel time is spent on matmuls with the remainder dominated by softmax and elementwise operations
- Amdahl's Law limits the overall speedup to ~1.20× even with a 2× matmul speedup. 

Speedups were consistently larger for forward+backward passes than forward-only, and grew modestly with model size, consistent with larger models being more compute-bound and better utilising tensor cores.